# Damage assessment — lesion area summary

This notebook **only reads existing** `damage_assessment/masks/<stem>/lesion_final.npy` files and the current `step2_masks/metadata_registration.csv`. It does not run the GUI, Step 4, or Step 7; it does not modify masks or calculate correlations in its primary area-summary section.

- Calculates lesion area in pixels and mm² at `PT`, `1-5h-PT`, `1d-PT`, and `3d-PT`. **`PT` is the first time point (within 1.5 h); `Day-PT` is baseline and has no lesion measurement.**
- Takes experimental groups **exclusively from** `analysis_selection.experimental_groups` in the active YAML; no manually defined mouse list.
- Applies `analysis_exclusions.enabled` and `analysis_exclusions.rules` **during summarization**, not during GUI annotation. Conditions in each rule's `match` mapping use AND across fields and OR within value lists.
- Produces an inclusion/selection audit for every discovered mask and an audit of exclusion rules, plus separate `stem`, `mouse × day`, and `group × day` tables and a mean ± SEM plot.

Hemorrhage masks, `diff_signed_505`, Step 7 outputs, and auxiliary maps **are not required**. A lesion mask is included even if that recording has no baseline. Missing masks are **not** assigned an area of zero.

**Aggregation convention:** when a mouse has multiple `stem` entries on the same day, its `mouse × day` area is the arithmetic mean of available valid masks, as in the original notebook. Pixel size is a user-specified parameter and must correspond to `rotated_fullres`.


## 1. Run settings

`CONFIG_PATH` must point to **your current YAML**. Experimental groups and exclusion rules are not defined in this cell. Verify the physical pixel scale: the `22.0` below was carried over from the original notebook and was not inferred from new images.


In [ ]:
from pathlib import Path

CONFIG_PATH = Path(r"config.yaml")

# Full-resolution pixel size in rotated_fullres coordinates (µm/pixel).
FULLRES_PIXEL_SIZE_UM = 22.0

# Explicit scientific time order. Day-PT is intentionally omitted because it is baseline.
TIME_ORDER = ["PT", "1-5h-PT", "1d-PT", "3d-PT"]
TIME_LABELS = {"PT": "PT (<1,5 ч)", "1-5h-PT": "1–5 ч", "1d-PT": "1 день", "3d-PT": "3 дня"}

SAVE_OUTPUTS = False  # Preserve earlier figures/CSV; only the separate ML/TTC section writes new files.
OUTPUT_FOLDER_NAME = "postprocess_summary"

## 2. Load YAML and project paths

Historical `F:` paths can be remapped through YAML `path_resolving` settings (for example, to `H:`). The input YAML is not rewritten.


In [ ]:
import json
import math
import warnings
from collections import OrderedDict
from pathlib import PureWindowsPath

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Config not found: {CONFIG_PATH}")
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8-sig"))
if not isinstance(config, dict):
    raise TypeError("YAML root must be a mapping")

def resolve_existing_project_root(cfg):
    """Resolve the existing analysis directory from YAML project and path mappings.
    
    Parameters
    ----------
    cfg : dict
        Project configuration with ``project`` and optional ``path_resolving``.
    
    Returns
    -------
    pathlib.Path
        First existing project directory among original and remapped paths.
    
    Raises
    ------
    FileNotFoundError
        If no candidate directory exists.
    """
    project = cfg.get("project", {}) or {}
    declared = str(project.get("directory_to_save_main_analysis", "")).strip()
    if not declared:
        raise ValueError("project.directory_to_save_main_analysis is not set")
    mapping = cfg.get("path_resolving", {}) or {}
    candidates = [Path(declared)]
    src_root, dst_root = mapping.get("from_root"), mapping.get("to_root")
    normalized = declared.replace("/", "\\")
    if src_root and dst_root:
        prefix = str(src_root).rstrip("\\/").replace("/", "\\")
        if normalized.lower() == prefix.lower() or normalized.lower().startswith(prefix.lower() + "\\"):
            suffix = normalized[len(prefix):].lstrip("\\")
            candidates.append(Path(str(PureWindowsPath(str(dst_root)) / PureWindowsPath(suffix))))
    src_drive, dst_drive = mapping.get("from_drive"), mapping.get("to_drive")
    wp = PureWindowsPath(normalized)
    if src_drive and dst_drive and wp.drive.lower() == str(src_drive).lower():
        target = PureWindowsPath(str(dst_drive).rstrip("\\/") + "\\")
        candidates.append(Path(str(target.joinpath(*wp.parts[1:]))))
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError("Analysis project directory is missing; tried: " + "; ".join(map(str, candidates)))

analysis_root = resolve_existing_project_root(config)
out_root = analysis_root / str((config.get("project", {}) or {}).get("out_folder_name", "out"))
damage_cfg = config.get("damage_assessment", {}) or {}
damage_root = out_root / str(damage_cfg.get("folder_name", "damage_assessment"))
masks_root = damage_root / str((damage_cfg.get("output", {}) or {}).get("masks_folder", "masks"))
step2_cfg = config.get("step2", {}) or {}
metadata_csv = (out_root / str(step2_cfg.get("folder_name", "step2_masks"))
                / str(step2_cfg.get("metadata_registration_name", "metadata_registration.csv")))
summary_root = damage_root / OUTPUT_FOLDER_NAME
summary_tables = summary_root / "tables"
summary_figures = summary_root / "figures"

if not masks_root.is_dir():
    raise FileNotFoundError(f"Lesion masks directory does not exist: {masks_root}")
if not metadata_csv.is_file():
    raise FileNotFoundError(f"Current Step2 metadata is required: {metadata_csv}")
if not np.isfinite(FULLRES_PIXEL_SIZE_UM) or FULLRES_PIXEL_SIZE_UM <= 0:
    raise ValueError("FULLRES_PIXEL_SIZE_UM must be positive and finite")
if len(set(TIME_ORDER)) != len(TIME_ORDER) or "Day-PT" in TIME_ORDER or "PT" not in TIME_ORDER:
    raise ValueError("TIME_ORDER must contain PT once and must not contain baseline Day-PT")

pixel_area_mm2 = (float(FULLRES_PIXEL_SIZE_UM) / 1000.0) ** 2
print("Project:", analysis_root)
print("Step2 metadata:", metadata_csv)
print("Lesion masks:", masks_root)
print("Output:", summary_root)
print("Time order:", TIME_ORDER)

## 3. Experimental groups from `experimental_groups`

`analysis_selection.experimental_groups` is the sole source of mouse-to-group assignments. If one mouse belongs to two groups, the notebook stops rather than selecting one arbitrarily. `analysis_selection.include_mice`, when provided, is also respected.


In [ ]:
def mouse_list(value):
    """Normalize a group membership specification into mouse ID strings.
    
    Parameters
    ----------
    value : dict, str, iterable, or None
        Explicit members or a mapping containing a supported membership key.
    
    Returns
    -------
    list of str
        Mouse identifiers in the supplied order, where applicable.
    """
    if isinstance(value, dict):
        for key in ("mice", "mouse_ids", "include_mice", "members"):
            if key in value:
                value = value[key]
                break
        else:
            raise ValueError(f"Unsupported experimental group specification: {value}")
    if value is None:
        return []
    if isinstance(value, str):
        return [value.strip()]
    if isinstance(value, (list, tuple, set)):
        return [str(x).strip() for x in value]
    raise TypeError(f"Mouse list must be a list or a string: {value!r}")

selection_cfg = config.get("analysis_selection", {}) or {}
raw_groups = selection_cfg.get("experimental_groups")
if not isinstance(raw_groups, dict) or not raw_groups:
    raise ValueError("Expected nonempty analysis_selection.experimental_groups mapping in current YAML")
analysis_groups = OrderedDict((str(name), mouse_list(spec)) for name, spec in raw_groups.items())
group_order = list(analysis_groups)
mouse_to_group = {}
for group_name, members in analysis_groups.items():
    for mouse in members:
        if mouse in mouse_to_group:
            raise ValueError(f"Mouse {mouse} occurs more than once: {mouse_to_group[mouse]}, {group_name}")
        mouse_to_group[mouse] = group_name
include_mouse_cfg = selection_cfg.get("include_mice")
include_mice = set(mouse_list(include_mouse_cfg)) if include_mouse_cfg is not None else None
require_group_assignment = bool(selection_cfg.get("require_group_assignment", False))
for group_name, members in analysis_groups.items():
    print(f"{group_name}: {len(members)} -> {', '.join(members)}")
print("Analysis include_mice:", sorted(include_mice) if include_mice is not None else "not restricted")

## 4. All masks and current metadata

Locate actual `lesion_final.npy` files by `stem`, rather than relying on an old `damage_gui_index.csv`, which may predate a change from `Day-PT_Awake` to `Day-PT_Iso`. Baseline is not required for this summary. Unrecognized `stem` values are excluded and recorded in the selection audit.


In [ ]:
registration = pd.read_csv(metadata_csv, low_memory=False)
for col in ("stem", "exp_day"):
    if col not in registration.columns:
        raise KeyError(f"Current Step2 metadata lacks required column {col!r}")
registration = registration.copy()
registration["stem"] = registration["stem"].astype("string")
if registration["stem"].isna().any() or registration["stem"].duplicated().any():
    raise ValueError("Step2 metadata contains empty or duplicated stems")
if "mouse_id" not in registration.columns:
    registration["mouse_id"] = registration["stem"].astype(str).str.split("_", n=1).str[0]
else:
    missing_mouse = registration["mouse_id"].isna() | registration["mouse_id"].astype(str).str.strip().eq("")
    registration.loc[missing_mouse, "mouse_id"] = registration.loc[missing_mouse, "stem"].astype(str).str.split("_", n=1).str[0]
registration["mouse_id"] = registration["mouse_id"].astype(str)
registration["exp_day"] = registration["exp_day"].astype(str)

mask_paths = sorted(masks_root.glob("*/lesion_final.npy"))
if not mask_paths:
    raise FileNotFoundError(f"No lesion_final.npy in {masks_root}")
mask_catalog = pd.DataFrame({
    "stem": [p.parent.name for p in mask_paths],
    "lesion_final_path": [str(p) for p in mask_paths],
})
if mask_catalog["stem"].duplicated().any():
    raise ValueError("Duplicate lesion stem folders")
meta_cols = [c for c in ("stem", "mouse_id", "group_id", "exp_day", "state", "protocol", "ref_stem", "baseline_stem")
             if c in registration.columns]
work = mask_catalog.merge(registration[meta_cols], on="stem", how="left", validate="one_to_one", indicator=True)
work["analysis_group"] = work["mouse_id"].map(mouse_to_group)
work["selection_status"] = "pending"
print("Found lesion masks:", len(work))
print("Matched to Step2 metadata:", int(work["_merge"].eq("both").sum()))

## 5. `analysis_exclusions`: strict selection and audit

Each `analysis_exclusions.rules` entry is evaluated against the **complete, current Step2 metadata**. Thus, a rule targeting only baseline `Day-PT` does not falsely appear unmatched just because baseline has no `lesion_final.npy`. `fail_on_unmatched_rules` is evaluated against that full metadata set. A `mouse_id + exp_day` rule excludes only the specified day; a whole-mouse rule affects all of that mouse's records.

Only masks for the target time points with an assigned experimental group and no matching exclusion rule enter the analysis.


In [ ]:
exclusions_cfg = config.get("analysis_exclusions", {}) or {}
apply_exclusions = bool(exclusions_cfg.get("enabled", False))
exclusion_rules = exclusions_cfg.get("rules", []) or []
if not isinstance(exclusion_rules, list):
    raise TypeError("analysis_exclusions.rules must be a list")

def match_rule(df, conditions):
    """AND across columns; OR inside list. Exact string matching, no implicit wildcards."""
    if not isinstance(conditions, dict) or not conditions:
        raise ValueError("Exclusion rule match must be a nonempty mapping")
    keep = pd.Series(True, index=df.index)
    for field, allowed in conditions.items():
        if field not in df.columns:
            raise KeyError(f"Exclusion rule mentions missing metadata field {field!r}")
        opts = allowed if isinstance(allowed, (list, tuple, set)) else [allowed]
        if not opts:
            raise ValueError(f"Exclusion rule field {field} has an empty match list")
        keep &= df[field].notna() & df[field].astype(str).isin([str(x) for x in opts])
    return keep

rule_audit_rows = []
exclusion_by_stem = {}
seen_rule_ids = set()
if apply_exclusions:
    for idx, rule in enumerate(exclusion_rules, 1):
        if not isinstance(rule, dict):
            raise TypeError(f"Exclusion rule #{idx} is not a mapping")
        rule_id = str(rule.get("id") or f"rule_{idx}")
        if rule_id in seen_rule_ids:
            raise ValueError(f"Repeated exclusion rule id: {rule_id}")
        seen_rule_ids.add(rule_id)
        conditions = rule.get("match")
        hit = match_rule(registration, conditions)
        matched = registration.loc[hit, "stem"].astype(str).tolist()
        reason = str(rule.get("reason", ""))
        for stem in matched:
            exclusion_by_stem.setdefault(stem, []).append((rule_id, reason))
        n_mask = int(work["stem"].isin(matched).sum())
        rule_audit_rows.append({
            "rule_id": rule_id, "reason": reason,
            "match_json": json.dumps(conditions, ensure_ascii=False, sort_keys=True),
            "matched_metadata_stems": int(len(matched)),
            "matched_lesion_masks": n_mask,
            "matched_target_day_masks": int((work["stem"].isin(matched) & work["exp_day"].isin(TIME_ORDER)).sum()),
            "status": "matched" if matched else "unmatched",
        })

rule_audit = pd.DataFrame(rule_audit_rows, columns=[
    "rule_id", "reason", "match_json", "matched_metadata_stems",
    "matched_lesion_masks", "matched_target_day_masks", "status",
])
if apply_exclusions and not rule_audit.empty and rule_audit["status"].eq("unmatched").any():
    names = rule_audit.loc[rule_audit["status"].eq("unmatched"), "rule_id"].tolist()
    message = f"analysis_exclusions rules unmatched in FULL Step2 metadata: {names}"
    if bool(exclusions_cfg.get("fail_on_unmatched_rules", False)):
        raise ValueError(message)
    warnings.warn(message)

work["exclusion_rule_ids"] = work["stem"].map(lambda s: ";".join(x[0] for x in exclusion_by_stem.get(str(s), [])))
work["exclusion_reasons"] = work["stem"].map(lambda s: ";".join(x[1] for x in exclusion_by_stem.get(str(s), [])))
work.loc[work["_merge"].ne("both"), "selection_status"] = "missing_step2_metadata"
work.loc[work["selection_status"].eq("pending") & ~work["exp_day"].isin(TIME_ORDER), "selection_status"] = "outside_target_days"
if include_mice is not None:
    mask = work["selection_status"].eq("pending") & ~work["mouse_id"].isin(include_mice)
    work.loc[mask, "selection_status"] = "outside_analysis_include_mice"
mask = work["selection_status"].eq("pending") & work["exclusion_rule_ids"].ne("")
work.loc[mask, "selection_status"] = "excluded_by_analysis_exclusions"
mask = work["selection_status"].eq("pending") & work["analysis_group"].isna()
work.loc[mask, "selection_status"] = "mouse_not_in_experimental_groups"
if require_group_assignment and mask.any():
    raise ValueError("Missing group assignment for target mouse(s): " + ", ".join(work.loc[mask, "mouse_id"].unique()))
work.loc[work["selection_status"].eq("pending"), "selection_status"] = "included"

selection_audit = work.drop(columns=["_merge"]).copy()
work_selected = work.loc[work["selection_status"].eq("included")].copy()
print("Exclusions enabled:", apply_exclusions)
print("Masks for analysis:", len(work_selected))
display(work["selection_status"].value_counts().rename_axis("status").reset_index(name="n_masks"))
display(rule_audit)

## 6. Lesion area at `stem` and `mouse × day` levels

The same physical pixel size is used for all records in this summary. Mask-loading failures are retained as `error` rows; they are not converted to zero-area lesions.


In [ ]:
stem_rows = []
for _, r in work_selected.iterrows():
    rec = r.drop(labels=["_merge"]).to_dict()
    rec["fullres_pixel_size_um"] = float(FULLRES_PIXEL_SIZE_UM)
    rec["pixel_area_mm2"] = float(pixel_area_mm2)
    try:
        lesion = np.load(Path(rec["lesion_final_path"]), allow_pickle=False)
        if lesion.ndim != 2:
            raise ValueError(f"Expected 2D lesion mask, got {lesion.shape}")
        if not np.issubdtype(lesion.dtype, np.number) and lesion.dtype != np.dtype("bool"):
            raise TypeError(f"Unexpected lesion mask dtype: {lesion.dtype}")
        if not np.isfinite(lesion).all():
            raise ValueError("lesion mask contains NaN/Inf")
        area_px = int(np.count_nonzero(lesion > 0))
        rec.update({"lesion_area_px": area_px, "lesion_area_mm2": area_px * pixel_area_mm2,
                    "lesion_shape_y": int(lesion.shape[0]), "lesion_shape_x": int(lesion.shape[1]),
                    "status": "ok", "error": ""})
    except Exception as exc:
        rec.update({"lesion_area_px": np.nan, "lesion_area_mm2": np.nan,
                    "lesion_shape_y": np.nan, "lesion_shape_x": np.nan,
                    "status": "error", "error": repr(exc)})
    stem_rows.append(rec)

stem_summary = pd.DataFrame(stem_rows)
if stem_summary.empty:
    stem_summary = pd.DataFrame(columns=["analysis_group", "mouse_id", "stem", "exp_day",
                                         "lesion_area_px", "lesion_area_mm2", "status", "error"])
valid_stems = stem_summary.loc[stem_summary["status"].eq("ok")].copy()
mouse_day_rows = []
for (group, mouse, day), g in valid_stems.groupby(["analysis_group", "mouse_id", "exp_day"], sort=False):
    mouse_day_rows.append({
        "analysis_group": str(group), "mouse_id": str(mouse), "exp_day": str(day),
        "n_stems": int(len(g)),
        "lesion_area_px": float(g["lesion_area_px"].mean()),
        "lesion_area_mm2": float(g["lesion_area_mm2"].mean()),
        "stems": ";".join(g["stem"].astype(str)),
    })
mouse_day = pd.DataFrame(mouse_day_rows, columns=["analysis_group", "mouse_id", "exp_day",
                                                  "n_stems", "lesion_area_px", "lesion_area_mm2", "stems"])

print("Mask calculation status:")
display(stem_summary["status"].value_counts(dropna=False).rename("n").to_frame())
if not mouse_day.empty and mouse_day["n_stems"].gt(1).any():
    warnings.warn("More than one mask stem for some mouse × day; mean area used (historical rule)")
    display(mouse_day.loc[mouse_day["n_stems"].gt(1)])
display(mouse_day.head(30))

## 7. Descriptive statistics by group and time point

Every configured group and time point is represented, including combinations with no valid masks (`n_mice = 0`, area `NaN`). This makes missing data visible without conflating them with the absence of a lesion.


In [ ]:
def numeric_stats(vals):
    """Compute descriptive statistics after excluding non-finite observations.
    
    Parameters
    ----------
    vals : array-like
        Mouse-level lesion areas, in the units of the selected metric.
    
    Returns
    -------
    dict
        Count, mean, sample SD, SEM, median, minimum, and maximum.
        SD and SEM are NaN when fewer than two observations remain.
    """
    x = pd.to_numeric(pd.Series(vals), errors="coerce")
    x = x[np.isfinite(x)]
    n = len(x)
    return {
        "n_mice": int(n),
        "mean": float(x.mean()) if n else np.nan,
        "sd": float(x.std(ddof=1)) if n >= 2 else np.nan,
        "sem": float(x.std(ddof=1) / math.sqrt(n)) if n >= 2 else np.nan,
        "median": float(x.median()) if n else np.nan,
        "min": float(x.min()) if n else np.nan,
        "max": float(x.max()) if n else np.nan,
    }

summary_rows = []
for metric in ("lesion_area_mm2", "lesion_area_px"):
    for group in group_order:
        for day in TIME_ORDER:
            if mouse_day.empty:
                g = pd.DataFrame(columns=[metric])
            else:
                g = mouse_day[(mouse_day["analysis_group"] == group) & (mouse_day["exp_day"] == day)]
            summary_rows.append({"analysis_group": group, "exp_day": day,
                                 "metric": metric, **numeric_stats(g[metric])})
area_group_day_summary = pd.DataFrame(summary_rows)
display(area_group_day_summary)

## 8. Lesion-area trajectory — mean ± SEM and individual mice

The first x-axis point represents `PT (<1.5 h)`. Time points without observations are left blank. SD and SEM are undefined for a single mouse; only its measured value is displayed.


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.5))
x_map = {day: i for i, day in enumerate(TIME_ORDER)}
for group in group_order:
    gm = mouse_day.loc[mouse_day["analysis_group"].eq(group)].copy()
    for mouse, one in gm.groupby("mouse_id", sort=False):
        one = one.copy()
        one["x"] = one["exp_day"].map(x_map)
        one = one.sort_values("x")
        ax.plot(one["x"], one["lesion_area_mm2"], marker="o", linewidth=1, alpha=0.22)
    grouped = area_group_day_summary.loc[
        area_group_day_summary["analysis_group"].eq(group)
        & area_group_day_summary["metric"].eq("lesion_area_mm2")
    ].copy()
    grouped["x"] = grouped["exp_day"].map(x_map)
    grouped = grouped.sort_values("x")
    good = grouped["mean"].notna()
    if good.any():
        yy = grouped.loc[good, "mean"].to_numpy(dtype=float)
        sem = grouped.loc[good, "sem"].fillna(0).to_numpy(dtype=float)
        ax.errorbar(grouped.loc[good, "x"].to_numpy(dtype=float), yy, yerr=sem,
                    marker="o", capsize=4, linewidth=2, label=group)
    else:
        ax.plot([], [], marker="o", label=group)
ax.set_xticks(range(len(TIME_ORDER)), [TIME_LABELS.get(d, d) for d in TIME_ORDER])
ax.set_xlabel("Время после фототромбоза")
ax.set_ylabel("Площадь очага, мм²")
ax.set_title("Динамика площади повреждения")
ax.grid(axis="y", alpha=0.2)
ax.legend(title="Экспериментальная группа")
fig.tight_layout()
if SAVE_OUTPUTS:
    summary_figures.mkdir(parents=True, exist_ok=True)
    fig.savefig(summary_figures / "lesion_area_dynamics_mm2.png", dpi=220, bbox_inches="tight")
plt.show()

## 9. Saving outputs and reproducibility

When `SAVE_OUTPUTS` is enabled, the notebook writes to the dedicated `damage_assessment/postprocess_summary/` directory. The original `lesion_final.npy` files, Step2 metadata, GUI indices, YAML, and upstream processing outputs are not modified.

- `lesion_stem_summary.csv` — all included `stem` values, including mask-read failures.
- `lesion_mouse_day_summary.csv` — one row per mouse and day.
- `lesion_group_day_summary.csv` — mean/SD/SEM/median by group and day.
- `lesion_selection_audit.csv` — **all discovered masks**, with inclusion or exclusion reasons.
- `analysis_exclusion_rule_audit.csv` — matches for every rule against complete Step2 metadata.
- `summary_settings.json` — summary settings and output provenance.

**Current notebook setting:** `SAVE_OUTPUTS = False`, so the primary summary files are not written. The later ML/TTC validation block has an **independent** saving flag (`SAVE_ML_TTC_OUTPUTS`).


In [ ]:
if SAVE_OUTPUTS:
    summary_tables.mkdir(parents=True, exist_ok=True)
    outputs = {
        "lesion_stem_summary.csv": stem_summary,
        "lesion_mouse_day_summary.csv": mouse_day,
        "lesion_group_day_summary.csv": area_group_day_summary,
        "lesion_selection_audit.csv": selection_audit,
        "analysis_exclusion_rule_audit.csv": rule_audit,
    }
    for filename, df in outputs.items():
        path = summary_tables / filename
        df.to_csv(path, index=False)
        print("Saved:", path)
    settings = {
        "source_config": str(CONFIG_PATH),
        "source_step2_metadata": str(metadata_csv),
        "masks_root": str(masks_root),
        "experimental_groups_source": "analysis_selection.experimental_groups",
        "experimental_groups": {group: members for group, members in analysis_groups.items()},
        "analysis_include_mice": sorted(include_mice) if include_mice is not None else None,
        "require_group_assignment": require_group_assignment,
        "analysis_exclusions_enabled": apply_exclusions,
        "analysis_exclusions_fail_on_unmatched_rules": bool(exclusions_cfg.get("fail_on_unmatched_rules", False)),
        "analysis_exclusion_rules": exclusion_rules if apply_exclusions else [],
        "time_order": TIME_ORDER,
        "time_labels": TIME_LABELS,
        "fullres_pixel_size_um": float(FULLRES_PIXEL_SIZE_UM),
        "pixel_area_mm2": float(pixel_area_mm2),
        "aggregation_multiple_stems_same_mouse_day": "arithmetic_mean_of_valid_masks",
        "baseline_required": False,
        "hemorrhage_evaluated": False,
        "n_mask_files_found": int(len(mask_catalog)),
        "n_stems_included": int(len(work_selected)),
        "n_stems_area_ok": int(len(valid_stems)),
    }
    settings_path = summary_root / "summary_settings.json"
    settings_path.write_text(json.dumps(settings, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    print("Saved:", settings_path)
    print("Saved:", summary_figures / "lesion_area_dynamics_mm2.png")
else:
    print("SAVE_OUTPUTS=False: no files written")

## 10. Final QC

Check mouse counts for every group and time point, unreadable masks, and excluded `stem` values. Before scientific interpretation, verify that the pixel size matches the current image geometry.


In [ ]:
print("=== Число мышей с валидными масками и средняя площадь ===")
display(area_group_day_summary.loc[
    area_group_day_summary["metric"].eq("lesion_area_mm2"),
    ["analysis_group", "exp_day", "n_mice", "mean", "sd", "sem", "median"],
])
print("\n=== Решения об исключениях / пропусках ===")
display(selection_audit.loc[
    selection_audit["selection_status"].ne("included"),
    [c for c in ["mouse_id", "stem", "exp_day", "analysis_group", "selection_status", "exclusion_rule_ids", "exclusion_reasons"]
     if c in selection_audit.columns],
].head(100))
print("\n=== Ошибки чтения масок ===")
display(stem_summary.loc[stem_summary["status"].eq("error"),
                        ["mouse_id", "stem", "exp_day", "lesion_final_path", "error"]].head(50))

---

## 11. Extension — anatomical ML diameter and TTC validation

This block is **independent of the preceding summary tables and figures**. It uses only `stem_summary` records **already included in the analysis** (after `experimental_groups`, `include_mice`, and `analysis_exclusions`), the original `lesion_final.npy` masks, the Step 3 AP axis, and the Step 2/3 registration actually applied.

**Definition:** the maximum continuous cross-sectional mask width perpendicular to the anatomical AP axis, estimated by subpixel discrete scanning with nearest-neighbor sampling. Disconnected fragments are not joined. Masks are not rotated, rescaled, or rewritten. Length in pixels is converted to mm using `FULLRES_PIXEL_SIZE_UM`.

The AP axis is read from `out/step3/reference_annotations.csv` (`axis_p1`, `axis_p2`) by the exact `group_id + ref_stem` pair. For the current `stem`, the axis is transferred using the inverse of the Step 2 **current → reference** transform, giving priority to the final Step 3 manual registration. Missing or invalid axes/transforms produce an error status; no default orientation is substituted.

**Protection of earlier outputs:** `SAVE_OUTPUTS = False` above prevents overwriting the original `lesion_area_dynamics_mm2.png`, summary CSV files, and `summary_settings.json`. The **new** outputs of this extension are saved under `postprocess_summary/ml_ttc_validation/`, controlled separately by `SAVE_ML_TTC_OUTPUTS` (currently `True`). This notebook does not rewrite the source lesion masks.


### 11.1. TTC parameters and measurement precision

Set the path to your TTC workbook. In the example workbook, the sheet is `TTC_EK`, with columns `mouse#`, `tpoint`, and `Диаметр инсульта, мм` (the latter is an existing data-column name). TTC measurements at `3d_PT` are selected. The WFI mask is also selected at `3d-PT` by default, yielding one WFI–TTC pair per mouse.

The subpixel step controls **sampling precision at the raster boundary**; it does not improve the physical resolution of the original mask. Differences smaller than one pixel should not automatically be interpreted as biologically meaningful.


In [ ]:
from scipy.ndimage import map_coordinates, label as nd_label
from scipy.stats import spearmanr
import re

# Replace with the location of the XLSX workbook in your project.
TTC_PATH = Path(r"H:\PT2021_final_analysis\PT_2021_TTC.xlsx")
TTC_SHEET = "TTC_EK"
TTC_MOUSE_COLUMN = "mouse#"
TTC_TIME_COLUMN = "tpoint"
TTC_DIAMETER_COLUMN = "Диаметр инсульта, мм"
TTC_TIME_VALUE = "3d_PT"
CORR_EXP_DAY = "3d-PT"  # WFI mask time point for comparison with TTC at 3d_PT

# Measure the maximum CONTINUOUS width along ML, perpendicular to AP.
AP_SAMPLE_STEP_PX = 0.5
ML_SAMPLE_STEP_PX = 0.25
SAVE_ML_TTC_OUTPUTS = True  # Write ONLY new files under the separate output directory
SAVE_ML_QC_IMAGES = True

ML_TTC_ROOT = summary_root / "ml_ttc_validation"
ML_TTC_TABLES = ML_TTC_ROOT / "tables"
ML_TTC_FIGURES = ML_TTC_ROOT / "figures"
ML_TTC_QC = ML_TTC_FIGURES / "qc_ml_diameter"

assert CORR_EXP_DAY in TIME_ORDER, "CORR_EXP_DAY must be one of the analyzed days"
assert AP_SAMPLE_STEP_PX > 0 and ML_SAMPLE_STEP_PX > 0
print("Source Step3 axis:", out_root / str((config.get("step3", {}) or {}).get("folder_name", "step3"))
      / str((config.get("step3", {}) or {}).get("reference_annotations_name", "reference_annotations.csv")))
print("ML/TTC output (new files only):", ML_TTC_ROOT)


### 11.2. Geometry: Step 3 AP axis and final registration

Step 3 landmark coordinates are in `rotated_fullres`. The stored transform maps the **current image into reference space**, so its **inverse** is used to transfer the AP axis into the native lesion mask. If a manual override exists but its final parameters are invalid, the notebook does not silently fall back to automatic registration.


In [ ]:
step3_cfg = config.get("step3", {}) or {}
step3_root = out_root / str(step3_cfg.get("folder_name", "step3"))
axis_csv = step3_root / str(step3_cfg.get("reference_annotations_name", "reference_annotations.csv"))
manual_cfg = step3_cfg.get("manual_registration", {}) or {}
overrides_csv = (step3_root / str(manual_cfg.get("folder_name", "manual_registration")) /
                 str(manual_cfg.get("overrides_csv_name", "manual_registration_overrides.csv")))

if not axis_csv.is_file():
    raise FileNotFoundError(f"Step3 reference AP-axis CSV not found: {axis_csv}")
axes_table = pd.read_csv(axis_csv, low_memory=False)
AXIS_COLUMNS = ["group_id", "ref_stem", "axis_p1_x", "axis_p1_y", "axis_p2_x", "axis_p2_y"]
missing_axis_columns = [col for col in AXIS_COLUMNS if col not in axes_table.columns]
if missing_axis_columns:
    raise KeyError(f"Step3 reference annotations lack {missing_axis_columns}")
manual_table = pd.read_csv(overrides_csv, low_memory=False) if overrides_csv.is_file() else pd.DataFrame()


def _finite_float(value, name):
    """Convert a value to float and reject non-finite transformation parameters."""
    try:
        x = float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{name} must be finite, got {value!r}") from exc
    if not np.isfinite(x):
        raise ValueError(f"{name} is not finite: {value!r}")
    return x


def _unique_latest_match(df, conditions, kind):
    """Find an unambiguous metadata record matching all supplied field values.
    
    Notes
    -----
    Despite the historical helper name, duplicate matches raise an error;
    no record is selected by timestamp or row order.
    """
    if df.empty:
        return None
    selected = df
    for col, val in conditions.items():
        if col not in selected.columns:
            raise KeyError(f"{kind} lacks {col!r}")
        selected = selected.loc[selected[col].astype(str).eq(str(val))]
    if selected.empty:
        return None
    if len(selected) > 1:
        # Step3 normally upserts unique group/stem rows. Multiple axis versions
        # are ambiguous for scientific measurement: do not guess.
        raise ValueError(f"{kind}: duplicate records for {conditions} (n={len(selected)})")
    return selected.iloc[0]


def _euclidean_matrix(rotation_rad, tx, ty):
    """Build a 3 × 3 homogeneous rotation-and-translation matrix.
    
    Parameters
    ----------
    rotation_rad : float
        In-plane rotation in radians.
    tx, ty : float
        Translation along x and y in full-resolution pixels.
    
    Returns
    -------
    numpy.ndarray
        Homogeneous XY transform with shape ``(3, 3)``.
    """
    theta = _finite_float(rotation_rad, "rotation_rad")
    c, s = np.cos(theta), np.sin(theta)
    return np.asarray([[c, -s, _finite_float(tx, "translation_x")],
                       [s,  c, _finite_float(ty, "translation_y")],
                       [0., 0., 1.]], dtype=float)


def _matrix_from_final_override(record):
    """Read the final manual registration as a validated XY transform matrix."""
    value = record.get("final_transform_xy_json", "")
    if pd.notna(value) and str(value).strip():
        try:
            mat = np.asarray(json.loads(str(value)), dtype=float)
        except Exception as exc:
            raise ValueError("malformed final_transform_xy_json") from exc
        if mat.shape != (3, 3) or not np.isfinite(mat).all():
            raise ValueError("final_transform_xy_json is not a finite 3×3 matrix")
        return mat
    return _euclidean_matrix(record.get("final_rotation_rad"),
                             record.get("final_translation_x"),
                             record.get("final_translation_y"))


def _current_to_ref_matrix(row):
    """Resolve the current-to-reference transform, prioritizing final manual overrides."""
    stem, ref_stem = str(row.get("stem", "")), str(row.get("ref_stem", ""))
    if not stem or not ref_stem or ref_stem.lower() == "nan":
        raise ValueError("stem or ref_stem missing from current Step2 metadata")
    if stem == ref_stem:
        return np.eye(3), "reference_identity"
    if bool(damage_cfg.get("input", {}).get("use_manual_registration_overrides", True)) and not manual_table.empty:
        cond = {"stem": stem}
        if "group_id" in manual_table.columns:
            cond["group_id"] = str(row.get("group_id", ""))
        # If there are historical duplicates, use the most recent manual save.
        hits = manual_table
        for col, val in cond.items():
            hits = hits.loc[hits[col].astype(str).eq(val)]
        if not hits.empty:
            if "updated_at" in hits.columns:
                hits = hits.sort_values("updated_at", kind="stable")
            mat = _matrix_from_final_override(hits.iloc[-1])
            source = "step3_manual_final"
        else:
            mat = _euclidean_matrix(row.get("rotation_rad"), row.get("translation_x"),
                                    row.get("translation_y"))
            source = "step2_auto"
    else:
        mat = _euclidean_matrix(row.get("rotation_rad"), row.get("translation_x"),
                                row.get("translation_y"))
        source = "step2_auto"
    if not np.isfinite(mat).all() or not np.allclose(mat[2], [0, 0, 1], atol=1e-6):
        raise ValueError("Invalid saved current→reference transform")
    A = mat[:2, :2]
    if not np.allclose(A.T @ A, np.eye(2), atol=1e-3) or not (0.99 < np.linalg.det(A) < 1.01):
        raise ValueError("Saved transform is not a rigid Euclidean transform")
    return mat, source


def _ap_axis_in_current(row):
    """Transfer the reference AP axis into native lesion-mask coordinates.
    
    Notes
    -----
    The inverse current-to-reference transform is used; coordinates are
    full-resolution XY pixels, not Allen atlas coordinates.
    """
    if "group_id" not in row or pd.isna(row.get("group_id")):
        raise ValueError("group_id is absent in current Step2 metadata")
    rec = _unique_latest_match(axes_table,
                               {"group_id": str(row["group_id"]),
                                "ref_stem": str(row["ref_stem"])}, "reference axis")
    if rec is None:
        raise ValueError(f"No reference AP axis for group_id={row['group_id']}, ref_stem={row['ref_stem']}")
    endpoints_ref = np.array([
        [_finite_float(rec["axis_p1_x"], "axis_p1_x"), _finite_float(rec["axis_p1_y"], "axis_p1_y")],
        [_finite_float(rec["axis_p2_x"], "axis_p2_x"), _finite_float(rec["axis_p2_y"], "axis_p2_y")],
    ], dtype=float)
    if np.linalg.norm(endpoints_ref[1] - endpoints_ref[0]) < 1:
        raise ValueError("Reference AP axis is too short")
    matrix, source = _current_to_ref_matrix(row)
    current_hom = (np.linalg.inv(matrix) @ np.column_stack([endpoints_ref, np.ones(2)]).T).T
    current = current_hom[:, :2] / current_hom[:, 2:3]
    delta = current[1] - current[0]
    length = np.linalg.norm(delta)
    if not np.isfinite(length) or length < 1:
        raise ValueError("Transformed AP axis is invalid or too short")
    ap = delta / length             # (x, y) in the current-mask coordinate system
    ml = np.array([-ap[1], ap[0]]) # unit vector perpendicular to AP
    return current, ap, ml, source


def measure_max_continuous_ml(mask, ap, ml, *, ap_step=0.5, ml_step=0.25):
    """Max contiguous cross-section of a raster mask along ML ⟂ AP.

    Raster nearest-neighbour sampling only; no resaving, rotation or connected
    component merging. Coordinates are pixel-centre (x,y). Width in pixels
    equals the number of inside samples × ml_step (subpixel approximation).
    """
    lesion = np.asarray(mask) > 0
    if lesion.ndim != 2 or not lesion.any():
        raise ValueError("Cannot measure an empty or non-2D lesion mask")
    yy, xx = np.nonzero(lesion)
    xy = np.column_stack([xx.astype(float), yy.astype(float)])
    ap_coords = xy @ ap
    ml_coords = xy @ ml
    center_ap = float(np.mean(ap_coords))
    ap_lines = np.arange(np.min(ap_coords) - 0.5, np.max(ap_coords) + 0.5 + ap_step/2, ap_step)
    ml_values = np.arange(np.min(ml_coords) - 1.5, np.max(ml_coords) + 1.5 + ml_step/2, ml_step)
    if ap_lines.size * ml_values.size > 45_000_000:
        raise ValueError("ML sampling grid is unexpectedly large; inspect mask geometry")
    best_n = 0
    best_dist = np.inf
    best_segment = None
    layer = lesion.astype(np.uint8)
    for ap_at in ap_lines:
        xy_line = ap[None, :] * ap_at + ml[None, :] * ml_values[:, None]
        hits = map_coordinates(layer, [xy_line[:, 1], xy_line[:, 0]], order=0,
                               mode="constant", cval=0, prefilter=False) > 0
        edges = np.diff(np.concatenate([[False], hits, [False]]).astype(np.int8))
        starts = np.flatnonzero(edges == 1)
        stops = np.flatnonzero(edges == -1)
        if starts.size == 0:
            continue
        lengths = stops - starts
        largest = int(np.max(lengths))
        # Ties are resolved by proximity to the lesion AP centroid (deterministic).
        dist = abs(float(ap_at) - center_ap)
        if largest > best_n or (largest == best_n and dist < best_dist):
            idx = int(np.flatnonzero(lengths == largest)[0])
            best_n, best_dist = largest, dist
            start, stop = int(starts[idx]), int(stops[idx])
            t0 = float(ml_values[start] - ml_step/2)
            t1 = float(ml_values[stop - 1] + ml_step/2)
            p0 = ap * ap_at + ml * t0
            p1 = ap * ap_at + ml * t1
            best_segment = (p0, p1)
    if best_segment is None or best_n == 0:
        raise ValueError("No contiguous ML segment could be measured")
    return {
        "diameter_px": float(best_n * ml_step),
        "endpoint_0_xy": np.asarray(best_segment[0], dtype=float),
        "endpoint_1_xy": np.asarray(best_segment[1], dtype=float),
        "ap_unit_xy": np.asarray(ap, dtype=float),
        "ml_unit_xy": np.asarray(ml, dtype=float),
    }


### 11.3. Measuring all **included** lesions and QC

If the AP axis is missing from Step 3 or transform parameters are unavailable, the record remains in the output table with an error status. QC images show the mask outline, AP axis (cyan), and maximum ML-width segment (red). Earlier PNG files are not opened or overwritten.


In [ ]:
ml_rows = []
ml_qc_payloads = []
for _, sr in stem_summary.iterrows():
    rec = {k: sr.get(k, np.nan) for k in ("analysis_group", "mouse_id", "group_id", "stem",
                                          "ref_stem", "exp_day", "state", "protocol", "lesion_final_path")}
    rec.update({"diameter_status": "pending", "diameter_error": "",
                "lesion_ml_diameter_px": np.nan, "lesion_ml_diameter_mm": np.nan,
                "ap_axis_angle_deg_current": np.nan, "transform_source": "",
                "n_lesion_components": np.nan,
                "ml_segment_x0": np.nan, "ml_segment_y0": np.nan,
                "ml_segment_x1": np.nan, "ml_segment_y1": np.nan})
    if sr.get("status") != "ok":
        rec.update({"diameter_status": "invalid_lesion_mask",
                    "diameter_error": str(sr.get("error", ""))})
        ml_rows.append(rec)
        continue
    try:
        mask = np.load(Path(sr["lesion_final_path"]), allow_pickle=False)
        if mask.ndim != 2 or not np.isfinite(mask).all():
            raise ValueError("Lesion mask is non-2D or contains NaN/Inf")
        binary = mask > 0
        if not binary.any():
            raise ValueError("Lesion mask is empty")
        # stem_summary intentionally contains only selected descriptive columns;
        # numeric registration fields live in the full CURRENT Step2 metadata.
        reg_hit = registration.loc[registration["stem"].astype(str).eq(str(sr["stem"]))]
        if len(reg_hit) != 1:
            raise ValueError(f"Stem does not map uniquely to current Step2 registration: {sr['stem']}")
        reg_row = reg_hit.iloc[0].copy()
        endpoints, ap, ml, transform_source = _ap_axis_in_current(reg_row)
        result = measure_max_continuous_ml(binary, ap, ml,
                                           ap_step=AP_SAMPLE_STEP_PX, ml_step=ML_SAMPLE_STEP_PX)
        x0, y0 = result["endpoint_0_xy"]
        x1, y1 = result["endpoint_1_xy"]
        ncomp = int(nd_label(binary)[1])
        rec.update({
            "diameter_status": "ok", "transform_source": transform_source,
            "lesion_ml_diameter_px": result["diameter_px"],
            "lesion_ml_diameter_mm": result["diameter_px"] * FULLRES_PIXEL_SIZE_UM / 1000.,
            "ap_axis_angle_deg_current": float(np.degrees(np.arctan2(ap[1], ap[0]))),
            "n_lesion_components": ncomp,
            "ml_segment_x0": x0, "ml_segment_y0": y0,
            "ml_segment_x1": x1, "ml_segment_y1": y1,
        })
        if SAVE_ML_QC_IMAGES:
            ml_qc_payloads.append((str(sr["stem"]), binary, endpoints.copy(), result, rec["lesion_ml_diameter_mm"], ncomp))
    except Exception as exc:
        rec.update({"diameter_status": "error", "diameter_error": f"{type(exc).__name__}: {exc}"})
    ml_rows.append(rec)

ml_stem = pd.DataFrame(ml_rows)
if ml_stem.empty:
    ml_stem = pd.DataFrame(columns=["analysis_group", "mouse_id", "group_id", "stem",
                                    "ref_stem", "exp_day", "diameter_status", "diameter_error",
                                    "lesion_ml_diameter_px", "lesion_ml_diameter_mm"])
ml_good = ml_stem.loc[ml_stem["diameter_status"].eq("ok")].copy()
ml_mouse_day = (ml_good.groupby(["analysis_group", "mouse_id", "exp_day"], as_index=False, sort=False)
                .agg(lesion_ml_diameter_px=("lesion_ml_diameter_px", "mean"),
                     lesion_ml_diameter_mm=("lesion_ml_diameter_mm", "mean"),
                     n_stems=("stem", "size"),
                     stems=("stem", lambda v: ";".join(map(str, v)))))
if not ml_mouse_day.empty and ml_mouse_day["n_stems"].gt(1).any():
    warnings.warn("Several included masks for a mouse × day: diameter averaged across valid stems, as for lesion area")
print("ML diameter calculation:")
display(ml_stem["diameter_status"].value_counts(dropna=False).to_frame("n"))
display(ml_mouse_day.head(30))
display(ml_stem.loc[ml_stem["diameter_status"].ne("ok"),
                     ["mouse_id", "stem", "exp_day", "diameter_error"]].head(50))


### 11.4. TTC comparison and Spearman correlation

There are **two independent reasons to exclude a pair**: no eligible WFI mask for the selected day (including exclusion by `analysis_exclusions`) and no TTC measurement. Experimental group assignments come **only from the current YAML**, not from Excel. Repeated TTC `mouse#` values at the selected time point are not averaged and are excluded from correlation automatically.


In [ ]:
def normalize_ttc_mouse(value):
    """Normalize a TTC spreadsheet mouse identifier to the ``m<number>`` form.
    
    Returns
    -------
    str or None
        Normalized mouse identifier, or None for an invalid or missing value.
    """
    if pd.isna(value):
        return None
    text = str(value).strip().lower()
    match = re.fullmatch(r"m_?0*(\d+)", text)
    if match:
        return f"m{int(match.group(1))}"
    return None

if not TTC_PATH.is_file():
    raise FileNotFoundError(f"Укажите путь к TTC XLSX в ячейке параметров: {TTC_PATH}")
try:
    ttc_raw = pd.read_excel(TTC_PATH, sheet_name=TTC_SHEET)
except ImportError as exc:
    raise ImportError("Для чтения XLSX через pandas нужен openpyxl в вашем окружении") from exc
required_ttc = [TTC_MOUSE_COLUMN, TTC_TIME_COLUMN, TTC_DIAMETER_COLUMN]
missing_ttc = [name for name in required_ttc if name not in ttc_raw.columns]
if missing_ttc:
    raise KeyError(f"TTC sheet {TTC_SHEET!r} missing columns: {missing_ttc}")
ttc_rows = ttc_raw.loc[ttc_raw[TTC_TIME_COLUMN].astype(str).str.strip().eq(TTC_TIME_VALUE)].copy()
ttc_rows["mouse_id"] = ttc_rows[TTC_MOUSE_COLUMN].map(normalize_ttc_mouse)
ttc_rows["ttc_diameter_mm"] = pd.to_numeric(ttc_rows[TTC_DIAMETER_COLUMN], errors="coerce")

# Mouse-level WFI values have ALREADY passed the notebook's group & exclusion rules.
wfi_compare = ml_mouse_day.loc[ml_mouse_day["exp_day"].eq(CORR_EXP_DAY)].copy()
if wfi_compare["mouse_id"].duplicated().any():
    raise ValueError("Multiple experimental groups for the same mouse at comparison day")

# Audit includes all TTC mice AND all WFI candidates, even if excluded or without mask.
all_possible_mice = set(ttc_rows["mouse_id"].dropna().astype(str))
all_possible_mice |= set(wfi_compare["mouse_id"].astype(str))
if "mouse_id" in registration.columns:
    all_possible_mice |= set(registration.loc[registration["exp_day"].eq(CORR_EXP_DAY), "mouse_id"].dropna().astype(str))

pair_rows = []
for mouse in sorted(all_possible_mice, key=lambda s: (int(s[1:]) if re.fullmatch(r"m\d+", s) else 10**9, s)):
    group = mouse_to_group.get(mouse, "")
    w = wfi_compare.loc[wfi_compare["mouse_id"].astype(str).eq(mouse)]
    t = ttc_rows.loc[ttc_rows["mouse_id"].astype(str).eq(mouse)]
    wfi_mm = float(w.iloc[0]["lesion_ml_diameter_mm"]) if len(w) == 1 else np.nan
    ttc_mm = float(t.iloc[0]["ttc_diameter_mm"]) if len(t) == 1 else np.nan
    audit = selection_audit.loc[(selection_audit["mouse_id"].astype(str).eq(mouse)) &
                                (selection_audit["exp_day"].astype(str).eq(CORR_EXP_DAY))]
    excluded = sorted(set(audit.loc[audit["selection_status"].ne("included"), "selection_status"].astype(str)))
    if len(w) == 0:
        status = "not_included_wfi" + (":" + ";".join(excluded) if excluded else "")
    elif len(t) == 0:
        status = "missing_ttc"
    elif len(t) != 1:
        status = "duplicate_ttc_rows"
    elif not np.isfinite(ttc_mm) or ttc_mm <= 0:
        status = "invalid_ttc_diameter"
    elif not np.isfinite(wfi_mm) or wfi_mm <= 0:
        status = "invalid_wfi_diameter"
    else:
        status = "included"
    pair_rows.append({
        "mouse_id": mouse, "analysis_group": group, "wfi_exp_day": CORR_EXP_DAY,
        "ttc_timepoint": TTC_TIME_VALUE, "wfi_ml_diameter_mm": wfi_mm,
        "ttc_diameter_mm": ttc_mm, "n_wfi_stems": int(w.iloc[0]["n_stems"]) if len(w) == 1 else 0,
        "n_ttc_rows": int(len(t)), "correlation_selection_status": status,
    })
ml_ttc_audit = pd.DataFrame(pair_rows)
ml_ttc_matched = ml_ttc_audit.loc[ml_ttc_audit["correlation_selection_status"].eq("included")].copy()

n_pairs = int(len(ml_ttc_matched))
if n_pairs >= 3 and (ml_ttc_matched["wfi_ml_diameter_mm"].nunique() >= 2) and (ml_ttc_matched["ttc_diameter_mm"].nunique() >= 2):
    result = spearmanr(ml_ttc_matched["wfi_ml_diameter_mm"].to_numpy(),
                       ml_ttc_matched["ttc_diameter_mm"].to_numpy(), alternative="two-sided")
    spearman_rho, spearman_p = float(result.statistic), float(result.pvalue)
    spearman_status = "ok"
else:
    spearman_rho, spearman_p = np.nan, np.nan
    spearman_status = "insufficient_pairs_or_constant_variable"
ml_ttc_spearman = pd.DataFrame([{
    "wfi_exp_day": CORR_EXP_DAY, "ttc_timepoint": TTC_TIME_VALUE,
    "n_mice": n_pairs, "spearman_rho": spearman_rho,
    "p_two_sided": spearman_p, "status": spearman_status,
    "groups_source": "analysis_selection.experimental_groups",
    "exclusions_enabled": apply_exclusions,
    "pixel_size_um": FULLRES_PIXEL_SIZE_UM,
    "ml_method": "max_contiguous_cross_section_perpendicular_to_Step3_AP_axis",
    "ap_step_px": AP_SAMPLE_STEP_PX, "ml_step_px": ML_SAMPLE_STEP_PX,
}])
print("Matched TTC/WFI pairs:", n_pairs)
display(ml_ttc_matched)
display(ml_ttc_spearman)
display(ml_ttc_audit.loc[ml_ttc_audit["correlation_selection_status"].ne("included")].head(60))


### 11.5. Scatterplot, QC images, and **separate** saving

The pre-existing `postprocess_summary/tables/*.csv` and `postprocess_summary/figures/lesion_area_dynamics_mm2.png` are not modified; new outputs go only to `ml_ttc_validation/`.

Each mouse is represented by one WFI–TTC point and colors are assigned by YAML group. **The code also draws an ordinary least-squares regression line when there are sufficient valid pairs**, optionally with 95% confidence and prediction bands. Spearman correlation and its bootstrap interval are calculated separately; neither is replaced by the linear fit. Earlier explanatory text incorrectly stated that no regression line was added.


In [ ]:
from scipy.stats import linregress, t as student_t
import numpy as np
import matplotlib.pyplot as plt

# =============================================
# WFI vs TTC: Spearman + regression visualization
# =============================================

SHOW_CONFIDENCE_BAND = True
SHOW_PREDICTION_BAND = True
SHOW_MOUSE_LABELS = True

pairs = ml_ttc_matched.copy()

x_col = "wfi_ml_diameter_mm"
y_col = "ttc_diameter_mm"

pairs[x_col] = pd.to_numeric(pairs[x_col], errors="coerce")
pairs[y_col] = pd.to_numeric(pairs[y_col], errors="coerce")

pairs = pairs.loc[
    np.isfinite(pairs[x_col]) &
    np.isfinite(pairs[y_col])
].copy()

fig, ax = plt.subplots(figsize=(7.5, 6.0))

# Experimental-group colors
colors_by_group = {
    name: plt.get_cmap("tab10")(i % 10)
    for i, name in enumerate(group_order)
}

# =============================================
# 1. Linear regression and intervals
# =============================================

x = pairs[x_col].to_numpy(dtype=float)
y = pairs[y_col].to_numpy(dtype=float)
n = len(x)

if n >= 3 and np.unique(x).size >= 2:

    fit = linregress(x, y)

    x_grid = np.linspace(x.min(), x.max(), 200)
    y_fit = fit.intercept + fit.slope * x_grid

    y_observed_fit = fit.intercept + fit.slope * x

    # Residual variance of the linear model
    df = n - 2
    residual_sd = np.sqrt(
        np.sum((y - y_observed_fit) ** 2) / df
    )

    x_mean = np.mean(x)
    Sxx = np.sum((x - x_mean) ** 2)

    t_crit = student_t.ppf(0.975, df)

    leverage = (
        1.0 / n +
        (x_grid - x_mean) ** 2 / Sxx
    )

    # 95% CI: uncertainty of the mean prediction
    ci = t_crit * residual_sd * np.sqrt(leverage)

    # 95% PI: variability of an individual observation
    pi = t_crit * residual_sd * np.sqrt(1.0 + leverage)

    # Draw the wider interval first
    if SHOW_PREDICTION_BAND:
        ax.fill_between(
            x_grid,
            y_fit - pi,
            y_fit + pi,
            color="#64748B",
            alpha=0.09,
            linewidth=0,
            label="95% prediction interval",
            zorder=1,
        )

    if SHOW_CONFIDENCE_BAND:
        ax.fill_between(
            x_grid,
            y_fit - ci,
            y_fit + ci,
            color="#64748B",
            alpha=0.23,
            linewidth=0,
            label="95% confidence interval",
            zorder=2,
        )

    ax.plot(
        x_grid,
        y_fit,
        color="#334155",
        linewidth=2.0,
        label="Linear regression (OLS)",
        zorder=3,
    )

# =============================================
# 2. Individual mouse points
# =============================================

for group in group_order:

    g = pairs.loc[
        pairs["analysis_group"].eq(group)
    ]

    if g.empty:
        continue

    ax.scatter(
        g[x_col],
        g[y_col],
        s=68,
        color=colors_by_group[group],
        edgecolor="white",
        linewidth=0.8,
        alpha=0.95,
        label=f"{group} (n={len(g)})",
        zorder=5,
    )

    if SHOW_MOUSE_LABELS:
        for _, r in g.iterrows():
            ax.annotate(
                str(r["mouse_id"]),
                (r[x_col], r[y_col]),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=8,
                alpha=0.85,
                zorder=6,
            )

# =============================================
# 3. Spearman correlation + bootstrap 95% CI
# =============================================

from scipy.stats import spearmanr

N_BOOTSTRAP = 10000
BOOTSTRAP_SEED = 42
CI_LEVEL = 0.95

rng = np.random.default_rng(BOOTSTRAP_SEED)

# x, y, and n are defined earlier in this cell
rho = np.nan
p_value = np.nan
rho_ci_low = np.nan
rho_ci_high = np.nan
bootstrap_values = np.array([])

if (
    n >= 4
    and np.unique(x).size >= 2
    and np.unique(y).size >= 2
):
    # Original Spearman correlation
    rho, p_value = spearmanr(x, y)

    bootstrap_rho = []

    for _ in range(N_BOOTSTRAP):

        # Bootstrap sampling of WFI–TTC pairs with replacement
        idx = rng.integers(0, n, size=n)

        xb = x[idx]
        yb = y[idx]

        # Skip degenerate bootstrap samples
        if (
            np.unique(xb).size < 2
            or np.unique(yb).size < 2
        ):
            continue

        rb, _ = spearmanr(xb, yb)

        if np.isfinite(rb):
            bootstrap_rho.append(rb)

    bootstrap_values = np.asarray(
        bootstrap_rho, dtype=float
    )

    if len(bootstrap_values) >= 1000:

        alpha = 1.0 - CI_LEVEL

        rho_ci_low, rho_ci_high = np.quantile(
            bootstrap_values,
            [alpha / 2, 1 - alpha / 2]
        )

# =============================================
# Results
# =============================================

stats_text = (
    f"Spearman ρ = {rho:.3f}\n"
    f"95% bootstrap CI: "
    f"[{rho_ci_low:.3f}, {rho_ci_high:.3f}]\n"
    f"p = {p_value:.3g}\n"
    f"n = {n}"
)

print(stats_text)

print(
    f"Valid bootstrap samples: "
    f"{len(bootstrap_values)}/{N_BOOTSTRAP}"
)

# Annotation on scatterplot
ax.text(
    0.03, 0.97,
    stats_text,
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=dict(
        boxstyle="round,pad=0.45",
        facecolor="white",
        edgecolor="#D1D5DB",
        alpha=0.92,
    ),
    zorder=10,
)

# =============================================
# 4. Styling
# =============================================

ax.set_xlabel(
    f"WFI: maximum ML lesion diameter ({CORR_EXP_DAY}), mm",
    fontsize=11,
)

ax.set_ylabel(
    f"TTC: lesion diameter ({TTC_TIME_VALUE}), mm",
    fontsize=11,
)

ax.set_title(
    "WFI vs TTC lesion diameter",
    fontsize=13,
    fontweight="bold",
    pad=12,
)

ax.grid(alpha=0.15, linewidth=0.7)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

if n:
    ax.legend(
        loc="best",
        fontsize=8,
        frameon=True,
    )
else:
    ax.text(
        0.5, 0.5,
        "No eligible WFI–TTC pairs",
        ha="center",
        va="center",
        transform=ax.transAxes,
    )

fig.tight_layout()

# =============================================
# 5. Save to a separate file
# =============================================

if SAVE_ML_TTC_OUTPUTS:
    ML_TTC_FIGURES.mkdir(parents=True, exist_ok=True)

    output_png = (
        ML_TTC_FIGURES /
        "lesion_ml_ttc_spearman_regression.png"
    )

    fig.savefig(
        output_png,
        dpi=300,
        bbox_inches="tight",
    )

    print("Saved:", output_png)

plt.show()

In [ ]:
def _native_qc_background(stem, shape):
    """Optional read-only Step2 native BMP beneath the mask; never required for QC."""
    try:
        import cv2
        hit = registration.loc[registration["stem"].astype(str).eq(str(stem))]
        if len(hit) != 1:
            return None
        row = hit.iloc[0]
        if str(row.get("stem")) == str(row.get("ref_stem")):
            fields = ("fixed_registration_path_resolved", "fixed_registration_path")
        else:
            fields = ("moving_registration_path_resolved", "moving_registration_path")
        cfg_paths = config.get("path_resolving", {}) or {}
        for col in fields:
            raw = row.get(col, "")
            if pd.isna(raw) or not str(raw).strip():
                continue
            candidates = [str(raw)]
            normalized = str(raw).replace("/", "\\")
            src_root, dst_root = cfg_paths.get("from_root"), cfg_paths.get("to_root")
            if src_root and dst_root:
                prefix = str(src_root).replace("/", "\\").rstrip("\\")
                if normalized.lower().startswith(prefix.lower() + "\\"):
                    candidates.append(str(PureWindowsPath(str(dst_root)) /
                                          PureWindowsPath(normalized[len(prefix):].lstrip("\\"))))
            src_drive, dst_drive = cfg_paths.get("from_drive"), cfg_paths.get("to_drive")
            wp = PureWindowsPath(normalized)
            if src_drive and dst_drive and wp.drive.lower() == str(src_drive).lower():
                candidates.append(str(PureWindowsPath(str(dst_drive).rstrip("\\/") + "\\").joinpath(*wp.parts[1:])))
            for candidate in candidates:
                if not Path(candidate).is_file():
                    continue
                bg = cv2.imread(str(candidate), cv2.IMREAD_UNCHANGED)
                if bg is None:
                    continue
                if bg.ndim == 3:
                    bg = cv2.cvtColor(bg, cv2.COLOR_BGR2GRAY)
                if tuple(bg.shape) == tuple(shape):
                    return np.asarray(bg, dtype=np.float32)
    except Exception:
        pass
    return None

# WFI versus TTC scatterplot: one point per mouse.
fig_corr, ax_corr = plt.subplots(figsize=(7.5, 6.0))
colors_by_group = {name: plt.get_cmap("tab10")(i % 10) for i, name in enumerate(group_order)}
for group in group_order:
    g = ml_ttc_matched.loc[ml_ttc_matched["analysis_group"].eq(group)]
    if g.empty:
        continue
    ax_corr.scatter(g["wfi_ml_diameter_mm"], g["ttc_diameter_mm"], s=63,
                    facecolor=colors_by_group[group], edgecolor="white", linewidth=0.65,
                    alpha=0.92, label=f"{group} (n={len(g)})")
    for _, r in g.iterrows():
        ax_corr.annotate(str(r["mouse_id"]), (r["wfi_ml_diameter_mm"], r["ttc_diameter_mm"]),
                         xytext=(4, 5), textcoords="offset points", fontsize=8, alpha=0.8)
ax_corr.set_xlabel(f"WFI: максимальный ML-диаметр ({CORR_EXP_DAY}), мм")
ax_corr.set_ylabel(f"TTC: диаметр очага ({TTC_TIME_VALUE}), мм")
ax_corr.set_title("ML-диаметр очага: WFI vs TTC")
ax_corr.grid(alpha=0.18)
ax_corr.text(0.03, 0.97, f"Spearman ρ = {spearman_rho:.3f}   p = {spearman_p:.3g}   n = {n_pairs}",
             transform=ax_corr.transAxes, ha="left", va="top", fontsize=10)
if len(ml_ttc_matched):
    ax_corr.legend(title="Экспериментальная группа", loc="best")
else:
    ax_corr.text(0.5, 0.5, "Нет допустимых пар WFI–TTC", transform=ax_corr.transAxes,
                 ha="center", va="center")
fig_corr.tight_layout()

if SAVE_ML_TTC_OUTPUTS:
    ML_TTC_TABLES.mkdir(parents=True, exist_ok=True)
    ML_TTC_FIGURES.mkdir(parents=True, exist_ok=True)
    new_tables = {
        "lesion_ml_diameter_stem.csv": ml_stem,
        "lesion_ml_diameter_mouse_day.csv": ml_mouse_day,
        "lesion_ml_ttc_matched.csv": ml_ttc_matched,
        "lesion_ml_ttc_selection_audit.csv": ml_ttc_audit,
        "lesion_ml_ttc_spearman.csv": ml_ttc_spearman,
    }
    for fn, table in new_tables.items():
        target = ML_TTC_TABLES / fn
        table.to_csv(target, index=False)
        print("Saved NEW:", target)
    fig_corr.savefig(ML_TTC_FIGURES / "lesion_ml_ttc_spearman.png", dpi=220, bbox_inches="tight")
    if SAVE_ML_QC_IMAGES:
        ML_TTC_QC.mkdir(parents=True, exist_ok=True)
        for stem, mask, endpoints, result, width_mm, ncomp in ml_qc_payloads:
            fig_qc, ax_qc = plt.subplots(figsize=(7, 6))
            ax_qc.set_facecolor("#252525")
            background = _native_qc_background(stem, mask.shape)
            if background is not None:
                inside = np.isfinite(background)
                vals = background[inside]
                low, high = np.nanpercentile(vals, [2, 98]) if vals.size else (0, 1)
                if not (np.isfinite(low) and np.isfinite(high) and high > low):
                    low, high = 0, float(np.nanmax(vals)) if vals.size else 1
                if high <= low: high = low + 1
                ax_qc.imshow(background, cmap="gray", interpolation="nearest",
                             origin="upper", vmin=low, vmax=high)
                ax_qc.imshow(np.ma.masked_where(~mask, mask), cmap="Reds", alpha=0.35,
                             interpolation="nearest", origin="upper", vmin=0, vmax=1)
            else:
                ax_qc.imshow(mask.astype(np.float32), cmap="gray", interpolation="nearest",
                             origin="upper", vmin=0, vmax=1)
            ax_qc.contour(mask.astype(float), levels=[0.5], colors=["#ffb703"], linewidths=0.75)
            center = endpoints.mean(axis=0)
            ap_vec = result["ap_unit_xy"]
            span = max(mask.shape) * 2
            p0 = center - span * ap_vec
            p1 = center + span * ap_vec
            ax_qc.plot([p0[0], p1[0]], [p0[1], p1[1]], color="#27b9e9",
                       linewidth=1.5, linestyle="--", label="AP (Step3)")
            a, b = result["endpoint_0_xy"], result["endpoint_1_xy"]
            ax_qc.plot([a[0], b[0]], [a[1], b[1]], color="white", linewidth=4.5)
            ax_qc.plot([a[0], b[0]], [a[1], b[1]], color="#eb1729", linewidth=2.8,
                       label=f"ML max: {width_mm:.3f} mm")
            ax_qc.scatter([a[0], b[0]], [a[1], b[1]], s=12, c="#eb1729")
            ax_qc.set_xlim(-0.5, mask.shape[1] - 0.5)
            ax_qc.set_ylim(mask.shape[0] - 0.5, -0.5)
            ax_qc.set_aspect("equal")
            ax_qc.set_title(f"{stem}  |  components={ncomp}")
            ax_qc.set_xlabel("X, pixels (rotated_fullres)")
            ax_qc.set_ylabel("Y, pixels (rotated_fullres)")
            ax_qc.legend(loc="upper right", fontsize=8)
            fig_qc.tight_layout()
            fig_qc.savefig(ML_TTC_QC / f"{stem}_ml_diameter_qc.png", dpi=150, bbox_inches="tight")
            plt.close(fig_qc)
    provenance = {
        "source_config": str(CONFIG_PATH), "source_step2_metadata": str(metadata_csv),
        "source_step3_axis_csv": str(axis_csv), "source_step3_manual_overrides_csv": str(overrides_csv),
        "source_ttc_xlsx": str(TTC_PATH), "ttc_sheet": TTC_SHEET,
        "wfi_day": CORR_EXP_DAY, "ttc_time": TTC_TIME_VALUE,
        "ml_method": "max_contiguous_perpendicular_to_ap_nearest_neighbor_sampling",
        "fullres_pixel_size_um": FULLRES_PIXEL_SIZE_UM,
        "ap_sampling_step_px": AP_SAMPLE_STEP_PX, "ml_sampling_step_px": ML_SAMPLE_STEP_PX,
        "analysis_group_source": "analysis_selection.experimental_groups",
        "exclusions_enabled": apply_exclusions,
        "original_summary_outputs_written": False,
    }
    (ML_TTC_ROOT / "ml_ttc_settings.json").write_text(
        json.dumps(provenance, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    print("ALL new outputs:", ML_TTC_ROOT)
else:
    print("SAVE_ML_TTC_OUTPUTS=False: no new files written")
plt.show()


### Interpretation

- **WFI diameter:** the maximum continuous lesion width along anatomical ML, perpendicular to the straight Step 3 AP axis.
- **TTC diameter:** a value read from the supplied Excel workbook without transformation. Verify that the TTC protocol measures a comparable anatomical diameter; correlations do not remove systematic methodological differences.
- Spearman correlation is computed **across mice, not recordings**, using only scientifically eligible pairs with both measurements.
- Inspect QC carefully for multi-component lesions (`n_lesion_components > 1`), strongly tilted AP axes, and unusually large values.
- `22 µm/pixel` is the current value carried over from the user's previous notebook, **not** a measured or independently confirmed pixel scale.
